# Notebook 3: Unsupervised Modeling — KMeans Clustering

In this notebook I train a KMeans clustering model on the cleaned dataset.
I use the Elbow Method and Silhouette Score to find the optimal number of clusters,
then assign a cluster label to each song.

In [ ]:
# Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

# Load the cleaned and scaled training data
df_scaled = pd.read_csv('../data/train_cleaned.csv')
df_ids = pd.read_csv('../data/train_ids.csv')

print("Cleaned data shape:", df_scaled.shape)
df_scaled.head()

## 1. Finding the Optimal Number of Clusters

### Method 1: Elbow Method


In [ ]:
import os

# Test k values from 2 to 12 and record inertia (sum of squared distances to cluster centers)
# Lower inertia = tighter clusters, but adding more clusters always reduces it
inertia_values = []
k_range = range(2, 13)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df_scaled)
    inertia_values.append(kmeans.inertia_)
    print(f"k={k}, inertia={kmeans.inertia_:.2f}")

# Plot — the 'elbow' bend suggests the best k before adding clusters stops helping
plt.figure(figsize=(9, 5))
plt.plot(k_range, inertia_values, marker='o', color='steelblue')
plt.title('Elbow Method — Finding Optimal Number of Clusters')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.xticks(k_range)
plt.tight_layout()
os.makedirs('../docs', exist_ok=True)
plt.savefig('../docs/elbow_plot.png')
plt.show()

### Method 2: Silhouette Score

The silhouette score measures how well each point fits in its own cluster vs. neighboring clusters.
A higher score means better-defined clusters.

In [ ]:
# Silhouette Score for each k
sil_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(df_scaled)
    score = silhouette_score(df_scaled, labels)
    sil_scores.append(score)
    print(f"k={k}, silhouette score={score:.4f}")

# Plot silhouette scores
plt.figure(figsize=(9, 5))
plt.plot(k_range, sil_scores, marker='o', color='darkorange')
plt.title('Silhouette Score by Number of Clusters')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.xticks(k_range)
plt.tight_layout()
plt.savefig('../docs/silhouette_plot.png')
plt.show()

## 2. Train Final Model with Optimal k

Based on the elbow and silhouette plots above, I'll select the best k and train the final model.

In [ ]:
# Pick the k with the highest silhouette score — best cluster separation
best_k = k_range[sil_scores.index(max(sil_scores))]
print(f"Best k based on silhouette score: {best_k}")

# Train the final model with the optimal k
# random_state=42 makes results reproducible
final_model = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = final_model.fit_predict(df_scaled)

print(f"\nCluster distribution:")
unique, counts = np.unique(cluster_labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Cluster {u}: {c} songs")

## 3. Attach Cluster Labels and Analyze

Now I'll attach the cluster labels back to the original song info and explore what each cluster looks like.

In [24]:
# Add cluster labels to the ids dataframe
df_ids['cluster'] = cluster_labels

# Also add back the original scaled features for analysis
df_result = pd.concat([df_ids, df_scaled], axis=1)

# Show sample songs from each cluster
for cluster in range(best_k):
    print(f"\n--- Cluster {cluster} Sample Songs ---")
    sample = df_result[df_result['cluster'] == cluster][['artist_name', 'track_name', 'genre']].head(5)
    print(sample.to_string(index=False))


--- Cluster 0 Sample Songs ---
        artist_name                      track_name genre
             mukesh            mohabbat bhi jhoothi   pop
stélios kazantzídis               finito la mouzika   pop
         ghantasala                 kanugona galano   pop
      mohammed rafi jahan men log sachhe ashikon ko   pop
    lata mangeshkar       maagata maagata janm gele   pop

--- Cluster 1 Sample Songs ---
  artist_name                    track_name genre
    liva weel  drømmer man om den, vågner..   pop
frankie laine                necessary evil   pop
  asha bhosle             gullyachi shapath   pop
talat mahmood ansoo to nahin hai ankhon men   pop
  a. m. rajah                 gopiparivrito   pop

--- Cluster 2 Sample Songs ---
            artist_name                               track_name genre
         the chordettes                            carolina moon   pop
        lata mangeshkar        paas nahin aaiye haath na lagaiye   pop
          andy williams it's the most wonde

In [ ]:
# Look at mean feature values per cluster to describe them
feature_cols = df_scaled.columns.tolist()
cluster_profiles = df_result.groupby('cluster')[feature_cols].mean()

# Heatmap of cluster feature profiles
plt.figure(figsize=(16, best_k + 2))
sns.heatmap(
    cluster_profiles,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    linewidths=0.5
)
plt.title('Mean Feature Values per Cluster')
plt.xlabel('Feature')
plt.ylabel('Cluster')
plt.tight_layout()
plt.savefig('../docs/cluster_profiles.png')
plt.show()

print(cluster_profiles)

## 4. Save the Clustered Training Data

In [ ]:
import os
import pickle

os.makedirs('../data', exist_ok=True)

# Save training data with cluster labels — used for analysis and recommendations in notebook 4
df_result.to_csv('../data/train_with_clusters.csv', index=False)
print("Saved train_with_clusters.csv")
print("Shape:", df_result.shape)

# Pickle the trained model so notebook 4 can load and apply it to test data
with open('../data/kmeans_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)

print("Saved kmeans_model.pkl")